<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Use Case 5 - Price Prediction
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<h2>Overview</h2>
<p></p>
<ol>
    <li>Import the required libraries and connect to Vantage</li>
    <li>Define functions to build and save the model</li>
    <ul>
        <li>Preprocess function</li>
        <li>Create tokenizer function</li>
        <li>Get train and test sets function</li>
        <li>Build the LSTM model function</li>
        <li>Predict function</li>
        <li>Save the tokenizer and save the model functions</li>
    </ul>
    <li>Define and run a main function</li>
    <li>Save the model in ONNX format</li>
    <li>Use the Teradata ONNXPredict function to run the model in the database</li>
    <li>Clean the output for visualization</li>
</ol>


<h3>1. Import the required libraries and connect to Vantage</h3>

In [1]:
# Import the required libraries
import pandas as pd
import numpy as np
import teradataml
from teradataml import DataFrame, create_context, execute_sql, configure, save_byom, ONNXPredict, retrieve_byom, remove_context
import settings
import json
import re
import time

#Create an LSTM model to predict the price based on the product description
import tensorflow as tf
import tf2onnx
from tf2onnx.convert import from_keras
from tensorflow.keras.preprocessing.text import Tokenizer, tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras import backend as K
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import joblib
import warnings
warnings.filterwarnings("ignore")

In [2]:
#Connect to the database
#remove_context()
connect_start_time = time.time()
create_context(host=settings.system_dsn, username=settings.system_user, password=settings.system_pw, logmech='LDAP')
connect_time = time.time() - connect_start_time

<b>Clear the session in the case of rerunning the notebook</b>

In [3]:
# Reset the session
K.clear_session()

<h3>2. Define functions to build and save the model</h3>

<h4>Define the pre process function</h4>
<p>The preprocess data function pulls the data from the database into a dataframe. The dataframe is then converted to a pandas dataframe and cleaned to ensure that the model can understand it.</p>

In [4]:
# Function to prepare data
def pre_process():
    #Load the data from the database
    select_query = """SELECT ProductID, ProductDescription, Price FROM marketplace;"""

    #Execute the select query
    df = DataFrame.from_query(select_query)

    df = df.to_pandas()

    #Clean the product descriptions
    df['ProductDescription'] = df['ProductDescription'].fillna('')

    return df

<h4>Define the create tokenizer function</h4>

In [5]:
# Function to create the tokenizer
def create_tokenizer(df):
    #Tokenize the text
    tokenizer = Tokenizer(num_words=10000, oov_token='<OOV>')
    tokenizer.fit_on_texts(df['ProductDescription'])
    return tokenizer

<h4>Define a function to get the train and test datasets</h4>
<p>Save the X_test data to a csv file for use later.</p>

In [6]:
# Function to get the train and test sets
def get_train_test_sets(tokenizer, df):
    #Convert the product descriptions to sequences
    sequences = tokenizer.texts_to_sequences(df['ProductDescription'])

    #Pad the sequences to the same length
    padded_sequences = pad_sequences(sequences, padding='post', maxlen=100)

    #Prepare the labels
    labels = df['Price'].values
    #Scale the labels
    scaler = MinMaxScaler()
    labels = scaler.fit_transform(labels.reshape(-1, 1)).flatten()
    joblib.dump(scaler, 'scaler.pkl')

    # Train and test split
    X_train, X_test, y_train, y_test = train_test_split(padded_sequences, labels, test_size=0.2, random_state=42)
    
    #Save the test data to a csv file
    #Create column names
    column_names = [f'  embedding_input_{i}' for i in range(100)]

    #Convert the test data to pandas
    df_test = pd.DataFrame(X_test, columns=column_names)

    #Save the test data to a csv file
    df_test.to_csv('marketplace_X_test.csv', index=False)

    return X_train, X_test, y_train, y_test


<h4>Define a function to build the LSTM model</h4>

In [7]:
# Function to create the model
def build_model(X_train, X_test, y_train, y_test):
    # Declare a model variable
    model = None

    #Build the LSTM model
    model = Sequential([
        Embedding(input_dim=10000, output_dim=64, input_length=100),
        LSTM(64, return_sequences=True),
        Dropout(0.3),
        LSTM(32),
        Dropout(0.3),
        Dense(32, activation='linear'),
        Dense(1)
    ])

    #Compile the model
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

    #Train the model

    #early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)
    #reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3)
    #, callbacks=[early_stop, reduce_lr]
    model.fit(X_train, y_train, epochs=22, validation_data=(X_test, y_test))

    return model


<h4>Define a function to predict on the test data</h4>

In [8]:
# Function to predict
def predict(X_test, y_test, model_4):
    predictions = model_4.predict(X_test[:5])

    for i in range(len(predictions)):
        print(f"Prediction: {predictions[i]}, Actual: {y_test[i]}")

<h4>Define functions to save the tokenizer and the model</h4>

In [9]:
# Save tokenizer function
def save_tokenizer(tokenizer):
    tokenizer_json = tokenizer.to_json()
    with open('tokenizer.json', 'w') as f:
        f.write(tokenizer_json)

In [10]:
# Save the model as ONNX file function
def save_model_as_onnx(model):
    
    #Save the model
    model.save("LSTM_model", save_format="tf")

    # Run a command line argument
    !python -m tf2onnx.convert --saved-model LSTM_model --output LSTM_model.onnx

<h3>3. Define and run the main function</h3>
<p>The main function runs all of the above functions building an LSTM model and saving it for future use.</p>

In [11]:
# Pre-process the data
pre_process_start_time = time.time()
df = pre_process()
pre_process_time = time.time() - pre_process_start_time
#print(f"Pre-processing time: {pre_process_time} seconds")

# Create and save tokenizer
tokenizer_start_time = time.time()
tokenizer = create_tokenizer(df)
save_tokenizer(tokenizer)
tokenizer_time = time.time() - tokenizer_start_time
#print(f"Tokenizer creation and saving time: {tokenizer_time} seconds")

# Get train and test sets
train_test_start_time = time.time()
X_train, X_test, y_train, y_test = get_train_test_sets(tokenizer, df)
train_test_time = time.time() - train_test_start_time
#print(f"Train/test split time: {train_test_time} seconds")

# Build and save model
model = None
build_model_start_time = time.time()
model = build_model(X_train, X_test, y_train, y_test)
build_model_time = time.time() - build_model_start_time
#print(f"Model building time: {build_model_time} seconds")

# Save the model
save_model_start_time = time.time()
save_model_as_onnx(model)
save_model_time = time.time() - save_model_start_time
#print(f"Model saving time: {save_model_time} seconds")

# Predict
predict_start_time = time.time()
predict(X_test, y_test, model)
predict_time = time.time() - predict_start_time
#print(f"Prediction time: {predict_time} seconds")

Epoch 1/22
1191/1191 [==============================] - 45s 36ms/step - loss: 0.0188 - mae: 0.0936 - val_loss: 0.0185 - val_mae: 0.0959
Epoch 2/22
1191/1191 [==============================] - 43s 36ms/step - loss: 0.0186 - mae: 0.0933 - val_loss: 0.0184 - val_mae: 0.0949
Epoch 3/22
1191/1191 [==============================] - 46s 39ms/step - loss: 0.0187 - mae: 0.0931 - val_loss: 0.0162 - val_mae: 0.0841
Epoch 4/22
1191/1191 [==============================] - 45s 38ms/step - loss: 0.0146 - mae: 0.0805 - val_loss: 0.0138 - val_mae: 0.0786
Epoch 5/22
1191/1191 [==============================] - 44s 37ms/step - loss: 0.0151 - mae: 0.0820 - val_loss: 0.0153 - val_mae: 0.0824
Epoch 6/22
1191/1191 [==============================] - 42s 35ms/step - loss: 0.0142 - mae: 0.0803 - val_loss: 0.0132 - val_mae: 0.0794
Epoch 7/22
1191/1191 [==============================] - 43s 36ms/step - loss: 0.0142 - mae: 0.0811 - val_loss: 0.0125 - val_mae: 0.0752
Epoch 8/22
1191/1191 [==========================

INFO:tensorflow:Assets written to: LSTM_model\assets


INFO:tensorflow:Assets written to: LSTM_model\assets
<frozen runpy>:128: RuntimeWarning: 'tf2onnx.convert' found in sys.modules after import of package 'tf2onnx', but prior to execution of 'tf2onnx.convert'; this may result in unpredictable behaviour
2025-09-04 13:34:52,283 - WARNING - ***IMPORTANT*** Installed protobuf is not cpp accelerated. Conversion will be extremely slow. See https://github.com/onnx/tensorflow-onnx/issues/1557
2025-09-04 13:34:52,283 - WARNING - '--tag' not specified for saved_model. Using --tag serve
2025-09-04 13:34:57,820 - INFO - Signatures found in model: [serving_default].
2025-09-04 13:34:57,830 - WARNING - '--signature_def' not specified, using first signature: serving_default
2025-09-04 13:34:57,830 - INFO - Output names: ['dense_1']
2025-09-04 13:34:58,333 - INFO - Using tensorflow=2.12.0, onnx=1.17.0, tf2onnx=1.16.1/15c810
2025-09-04 13:34:58,333 - INFO - Using opset <onnx, 15>
2025-09-04 13:34:58,403 - INFO - Computed 0 values for constant folding
202

1/1 [==============================] - 1s 659ms/step
Prediction: [0.0371554], Actual: 0.06372868965877346
Prediction: [0.04579528], Actual: 0.023627392110278568
Prediction: [0.1684491], Actual: 0.17732292509725087
Prediction: [0.02748439], Actual: 0.0172310262903689
Prediction: [0.04844867], Actual: 0.10803331331749474


<hr>
<h3>4. Save the model in ONNX format</h3>

<h4>Save model in database</h4>

In [12]:
#Start timer
load_model_start_time = time.time()

# Set BYOM install location
configure.byom_install_location = "mldb"

# Save the model path and name
model_path = 'LSTM_model.onnx'
model_name = 'LSTM_model'

#Drop the byom_models table if it exists
execute_sql("DROP TABLE byom_models;")

# Save the model to vantage
save_byom(model_name,
          model_file=model_path,
          table_name="byom_models")

# Bring in tokenizer
with open('tokenizer.json', 'r') as f:
    tokenizer_json = f.read()
tokenizer = tokenizer_from_json(tokenizer_json)

#Save time
load_model_time = time.time() - load_model_start_time

Created the model table 'byom_models' as it does not exist.
Model is saved.


<h4>Create a table of test data to run the saved model on</h4>
<p>Using the csv file that was saved while splitting the data into train and test sets, use Teradata FastLoad to load the data into a data table called marketplace_X_test in the TPCXAI database.</p>

<h3>5. Use the Teradata ONNXPredict function on the saved model</h3>

In [13]:
#Start the timer
start_time = time.time()

#Run the predict function
result = ONNXPredict (
    accumulate = [f"embedding_input_{i}" for i in range(100)],
    newdata=DataFrame.from_query("SELECT * FROM TPCXAI.marketplace_X_test"),
    modeldata=retrieve_byom(model_name, table_name="byom_models"),
    #show_model_input_fields_map = True,
    is_debug=True
)

#Stop the timer
end_time = time.time()

#Calculate the time taken
onnx_predict_time_taken = end_time - start_time

#Print the time taken
print(f"Time taken for ONNX prediction: {onnx_predict_time_taken} seconds")

Time taken for ONNX prediction: 10.278987646102905 seconds


In [14]:
print(result.result)

   embedding_input_0  embedding_input_1  embedding_input_2  embedding_input_3  embedding_input_4  embedding_input_5  embedding_input_6  embedding_input_7  embedding_input_8  embedding_input_9  embedding_input_10  embedding_input_11  embedding_input_12  embedding_input_13  embedding_input_14  embedding_input_15  embedding_input_16  embedding_input_17  embedding_input_18  embedding_input_19  embedding_input_20  embedding_input_21  embedding_input_22  embedding_input_23  embedding_input_24  embedding_input_25  embedding_input_26  embedding_input_27  embedding_input_28  embedding_input_29  embedding_input_30  embedding_input_31  embedding_input_32  embedding_input_33  embedding_input_34  embedding_input_35  embedding_input_36  embedding_input_37  embedding_input_38  embedding_input_39  embedding_input_40  embedding_input_41  embedding_input_42  embedding_input_43  embedding_input_44  embedding_input_45  embedding_input_46  embedding_input_47  embedding_input_48  embedding_input_49  embeddi

<h3>6. Format the output of the predict function to make sense of the output</h3>

In [15]:
#Function to convert the json_report to a single value
def convert_json_report(json_string):
    data = json.loads(json_string)
    number = data["dense_1"][0][0]
    return number

In [16]:
# Start time
convert_output_start_time = time.time()

# Save the result to a DataFrame
result_df = result.result.to_pandas()
#result_df.head()

result_data = []

for j in range(len(result_df)):
    result_line = str(result_df[f"embedding_input_0"].values[j])
    for i in range(1, 100):
        result_line += ', ' + str(result_df[f"embedding_input_{i}"].values[j])
    prediction = convert_json_report(result_df['json_report'].values[j])
    result_data.append((result_line, prediction))

result_data_df = pd.DataFrame(result_data)

# Convert the tokenized data back to text and the normalized prediction back to price
# Use the tokenizer to reverse
loaded_scaler = joblib.load('scaler.pkl')
for i in range(len(result_data_df)):
    result_data_df[0][i] = tokenizer.sequences_to_texts([list(map(int, result_data_df[0][i].split(', ')))])[0]
    result_data_df[1][i] = loaded_scaler.inverse_transform([[result_data_df[1][i]]])[0][0]

convert_data_time = time.time() - convert_output_start_time
result_data_df.head()

,0,1
0,viseart viseart eyeshadow palette new in box n...,52.243361
1,under armour <OOV> worn but still in stellar c...,49.581243
2,imperial <OOV> 9 imperial <OOV> arms legs good...,65.879928
3,tavik swimwear bikini bottoms brand new with n...,55.175904
4,heartsoul black nude lace dress sleeveless dre...,60.696385


<h5>The time it takes to run each individual step</h5>

In [17]:
#Print the time taken for all steps
print(f"Connect to database time:............ {connect_time} seconds")
print(f"Pre-processing time:................. {pre_process_time} seconds")
print(f"Tokenizer creation and saving time:.. {tokenizer_time} seconds")
print(f"Train/test split time:............... {train_test_time} seconds")
print(f"Model building time:................. {build_model_time} seconds or {build_model_time/60} minutes")
print(f"Model saving time:................... {save_model_time} seconds")
print(f"Test prediction time:................ {predict_time} seconds")
print(f"Load model time:..................... {load_model_time} seconds")
print(f"ONNX prediction time:................ {onnx_predict_time_taken} seconds")
print(f"Convert data time:................... {convert_data_time} seconds")

Connect to database time:............ 7.892160177230835 seconds
Pre-processing time:................. 16.162415981292725 seconds
Tokenizer creation and saving time:.. 0.8165426254272461 seconds
Train/test split time:............... 0.716956615447998 seconds
Model building time:................. 954.5856592655182 seconds or 15.909760987758636 minutes
Model saving time:................... 22.034024953842163 seconds
Test prediction time:................ 0.6837096214294434 seconds
Load model time:..................... 7.516920566558838 seconds
ONNX prediction time:................ 10.278987646102905 seconds
Convert data time:................... 15.336874008178711 seconds
